<a href="https://colab.research.google.com/github/karthikeya-20060626/Customer-retention-strategy-/blob/main/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist Supply Chain & Delivery Performance Analysis
**Portfolio project — operational diagnostic of a Brazilian e-commerce logistics network (2016–2018)**

**Scenario.** A private-equity firm has rolled up several small Brazilian logistics
partners and wants to standardise operations, lift delivery reliability, protect
margin, and find the underperforming hubs. This notebook is the analytical engine
behind that diagnostic: it cleans the raw Olist data, builds delivery KPIs, scores
sellers and states on risk, runs a root-cause decomposition, and sizes the value
at stake.

**Pipeline:** load → clean → build item-level fact → KPIs → seller matrix → risk
score → root cause → impact scenario.

> Data: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
DATA = '../data/'   # adjust if needed

## 1. Load raw tables

In [ ]:
orders    = pd.read_csv(DATA+'olist_orders.csv', parse_dates=[
    'order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
    'order_delivered_customer_date','order_estimated_delivery_date'])
items     = pd.read_csv(DATA+'olist_order_items.csv')
sellers   = pd.read_csv(DATA+'olist_sellers.csv')
products  = pd.read_csv(DATA+'olist_products.csv')
reviews   = pd.read_csv(DATA+'olist_order_reviews.csv')
customers = pd.read_csv(DATA+'olist_customers.csv')
trans     = pd.read_csv(DATA+'product_category_name_translation.csv')

print('orders', orders.shape, '| items', items.shape, '| sellers', sellers.shape)
orders['order_status'].value_counts()

orders (99441, 8) | items (112650, 7) | sellers (3095, 4)


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## 2. Clean — restrict to delivered orders, engineer delivery measures

Only `delivered` orders carry the full timestamp chain we need. We drop the
handful with missing delivery/estimate timestamps, then derive:

- **delivery_days** = purchase → delivered
- **processing_days** = purchase → handed to carrier
- **transit_days** = carrier → delivered
- **delay_days** = delivered − estimated  *(positive = late)*
- **is_late** = 1 when delivered after the estimate

> Note: the raw mean of `delay_days` is **negative** — Olist sets padded,
> conservative estimates, so most orders arrive early. That makes **late %**
> (not average delay) the honest reliability KPI.

In [ ]:
d = orders[orders['order_status']=='delivered'].copy()
d = d.dropna(subset=['order_delivered_customer_date','order_estimated_delivery_date',
                     'order_purchase_timestamp','order_delivered_carrier_date'])

day = lambda a,b: (a-b).dt.total_seconds()/86400
d['delivery_days']   = day(d['order_delivered_customer_date'], d['order_purchase_timestamp'])
d['processing_days'] = day(d['order_delivered_carrier_date'],  d['order_purchase_timestamp'])
d['transit_days']    = day(d['order_delivered_customer_date'], d['order_delivered_carrier_date'])
d['estimated_days']  = day(d['order_estimated_delivery_date'], d['order_purchase_timestamp'])
d['delay_days']      = day(d['order_delivered_customer_date'], d['order_estimated_delivery_date'])
d['is_late']         = (d['order_delivered_customer_date'] > d['order_estimated_delivery_date']).astype(int)

print(f'Delivered orders analysed: {len(d):,}')
print(f"On-time: {100*(1-d['is_late'].mean()):.1f}%  |  Late: {100*d['is_late'].mean():.1f}%")
print(f"Avg delivery {d['delivery_days'].mean():.1f}d | processing {d['processing_days'].mean():.1f}d | transit {d['transit_days'].mean():.1f}d")

Delivered orders analysed: 96,469
On-time: 91.9%  |  Late: 8.1%
Avg delivery 12.6d | processing 3.2d | transit 9.3d


## 3. Build the item-level fact table (joins seller, product, review, customer)

In [ ]:
# english category + size + volume
prod = products.merge(trans, on='product_category_name', how='left')
prod['category'] = prod['product_category_name_english'].fillna(prod['product_category_name']).fillna('unknown')
prod['volume_cm3'] = prod['product_length_cm']*prod['product_height_cm']*prod['product_width_cm']

rev = reviews.groupby('order_id')['review_score'].mean().reset_index()

f = (items
     .merge(d[['order_id','customer_id','order_purchase_timestamp','delivery_days','delay_days',
               'is_late','processing_days','transit_days']], on='order_id', how='inner')
     .merge(sellers, on='seller_id', how='left')
     .merge(prod[['product_id','category','product_weight_g','volume_cm3']], on='product_id', how='left')
     .merge(rev, on='order_id', how='left')
     .merge(customers[['customer_id','customer_state']], on='customer_id', how='left'))
f['revenue'] = f['price'] + f['freight_value']
f['cross_state'] = (f['seller_state'] != f['customer_state']).astype(int)
print(f'Fact rows (order items): {len(f):,}')
f.head(3)

Fact rows (order items): 110,188


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_purchase_timestamp,delivery_days,delay_days,is_late,processing_days,transit_days,seller_zip_code_prefix,seller_city,seller_state,category,product_weight_g,volume_cm3,review_score,customer_state,revenue,cross_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,7.614421,-8.011250,0,6.399468,1.214954,27277,volta redonda,SP,cool_stuff,650.0,3528.0,5.0,RJ,72.19,1
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,16.216181,-2.330278,0,8.154097,8.062083,3471,sao paulo,SP,pet_shop,30000.0,60000.0,4.0,SP,259.83,0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,7.948437,-13.444954,0,1.918947,6.029491,37564,borda da mata,MG,furniture_decor,3050.0,14157.0,5.0,MG,216.87,0


## 4. Delivery KPIs by seller state & category

In [ ]:
state = (f.groupby('seller_state')
           .agg(items=('order_item_id','count'), late_pct=('is_late','mean'),
                avg_delivery=('delivery_days','mean'), avg_transit=('transit_days','mean'),
                avg_review=('review_score','mean'), revenue=('revenue','sum'))
           .reset_index())
state['late_pct'] *= 100
state[state['items']>=200].sort_values('late_pct', ascending=False).head(10).round(2)

,seller_state,items,late_pct,avg_delivery,avg_transit,avg_review,revenue
6,MA,402,23.63,17.74,12.81,4.02,48169.50
21,SP,78598,8.52,12.28,8.98,4.05,9955994.13
15,RJ,4685,8.11,12.02,9.12,4.17,911918.67
3,DF,883,6.68,12.53,9.46,4.07,112892.78
4,ES,364,6.59,12.88,10.33,4.07,58853.49
14,PR,8486,6.45,13.38,9.88,4.13,1423967.27
19,SC,4000,5.88,13.57,10.44,4.13,717739.37
7,MG,8601,5.55,12.79,9.55,4.16,1184111.40
1,BA,624,5.45,13.85,10.45,4.16,297146.93
18,RS,2169,4.47,11.56,8.06,4.25,429749.09


## 5. Customer impact — does late delivery actually hurt reviews? (Yes.)

In [ ]:
do = f.groupby('order_id').agg(is_late=('is_late','max'), review=('review_score','mean')).dropna()
print('Avg review  | on-time: %.2f  | late: %.2f' % (
    do[do.is_late==0].review.mean(), do[do.is_late==1].review.mean()))
print('1-2 star %% | on-time: %.1f%%  | late: %.1f%%' % (
    100*(do[do.is_late==0].review<=2).mean(), 100*(do[do.is_late==1].review<=2).mean()))

Avg review  | on-time: 4.29  | late: 2.57
1-2 star % | on-time: 9.2%  | late: 54.0%


## 6. Seller Performance Matrix (4 quadrants)

In [ ]:
s = (f.groupby('seller_id')
       .agg(items=('order_item_id','count'), late_pct=('is_late','mean'),
            avg_review=('review_score','mean'), revenue=('revenue','sum'),
            state=('seller_state','first')).reset_index())
s['late_pct'] *= 100
VOL = s['items'].median(); LATE = 8.11
def quad(r):
    hv, good = r['items']>=VOL, r['late_pct']<=LATE
    return ('1 Strategic Partner' if hv and good else
            '2 Fix Immediately'   if hv and not good else
            '3 Monitor'           if not hv and not good else '4 Maintain')
s['quadrant'] = s.apply(quad, axis=1)
q = s.groupby('quadrant').agg(sellers=('seller_id','count'), revenue=('revenue','sum'),
                              avg_late=('late_pct','mean')).reset_index()
q['rev_pct'] = (100*q['revenue']/q['revenue'].sum()).round(1)
q.round(1)

,quadrant,sellers,revenue,avg_late,rev_pct
0,1 Strategic Partner,912,8202356.9,2.7,53.2
1,2 Fix Immediately,609,6273393.9,15.9,40.7
2,3 Monitor,271,271189.6,46.9,1.8
3,4 Maintain,1178,671260.5,0.0,4.4


## 7. Supply-Chain Risk Score (0–100)

Weighted min-max blend of late %, average delay when late, inverted review score,
and log-volume (exposure). A volume floor keeps the ranking actionable.

In [ ]:
def mm(x): return (x-x.min())/(x.max()-x.min()+1e-9)
risk = s.copy()
risk['avg_delay_late'] = (f[f.is_late==1].groupby('seller_id')['delay_days'].mean()
                          .reindex(risk['seller_id']).values)
risk['avg_delay_late'] = risk['avg_delay_late'].fillna(0)
risk['risk_score'] = (0.35*mm(risk['late_pct']) + 0.20*mm(risk['avg_delay_late'])
                    + 0.25*mm(5-risk['avg_review'].fillna(risk['avg_review'].median()))
                    + 0.20*mm(np.log1p(risk['items'])))*100
risk[risk['items']>=50].sort_values('risk_score', ascending=False).head(10).round(1)

,seller_id,items,late_pct,avg_review,revenue,state,quadrant,avg_delay_late,risk_score
1000,54965bbe3e4f07ae045b90b0b8541f52,81,32.1,3.1,12428.0,PR,2 Fix Immediately,28.0,37.4
478,2a1348e9addc1af5aaa619b1a3679d6b,51,29.4,3.0,3424.8,MG,2 Fix Immediately,39.9,36.7
323,1ca7077d890b907f89be8c954a02686a,127,19.7,2.3,13996.6,SP,2 Fix Immediately,4.9,36.6
1928,a49928bcdf77c55c6d6e05e09a9b4ca5,104,25.0,3.0,10373.8,SP,2 Fix Immediately,8.3,33.9
1480,7c67e1448b00f6e969d365cea6b010ab,1355,9.6,3.3,237806.7,SP,2 Fix Immediately,10.7,33.8
540,2eb70248d66e0e3ef83659f71b244378,197,14.2,2.8,42046.7,SP,2 Fix Immediately,7.7,32.9
1606,88460e8ebdecbfecb5f9601833981930,300,19.7,3.4,36258.6,PR,2 Fix Immediately,8.8,32.6
858,4a3ca9315b744ce9f8e9374361493884,1949,11.0,3.8,231220.4,SP,2 Fix Immediately,10.5,32.3
2171,bbad7e518d7af88a0897397ffdca1979,84,22.6,3.0,5741.2,SP,2 Fix Immediately,10.3,32.2
1139,602044f2c16190c2c6e45eb35c2e21cb,55,23.6,3.0,4468.9,SP,2 Fix Immediately,8.0,31.1


## 8. Root-cause decomposition

Four candidate drivers tested: lead-time split, geography, product size, and
seller effect within a single state.

In [ ]:
print('Lead time: processing %.1fd + transit %.1fd  (transit = %.0f%% of total)' % (
    f['processing_days'].mean(), f['transit_days'].mean(),
    100*f['transit_days'].mean()/(f['processing_days'].mean()+f['transit_days'].mean())))

print('\nGeography (same vs cross-state late %):')
print((f.groupby('cross_state')['is_late'].mean()*100).round(2).to_string())

f['wq'] = pd.qcut(f['product_weight_g'].fillna(f['product_weight_g'].median()), 4,
                  labels=['light','q2','q3','heavy'])
print('\nProduct weight quartile late %:')
print((f.groupby('wq', observed=True)['is_late'].mean()*100).round(2).to_string())

sp = f[f.seller_state=='SP'].groupby('seller_id')['is_late'].agg(['mean','count'])
sp = sp[sp['count']>=100]
print('\nSeller effect within SP — late%% p10/median/p90: %.1f / %.1f / %.1f' % (
    sp['mean'].quantile(.1)*100, sp['mean'].median()*100, sp['mean'].quantile(.9)*100))

Lead time: processing 3.3d + transit 9.2d  (transit = 74% of total)

Geography (same vs cross-state late %):
cross_state
0    5.98
1    9.00

Product weight quartile late %:
wq
light    7.36
q2       7.60
q3       8.18
heavy    8.57



Seller effect within SP — late% p10/median/p90: 4.0 / 8.1 / 13.8


**Read-out.** Transit (last-mile) is ~74% of lead time; cross-state routes are
~50% more likely to be late; product weight matters only weakly; and late rates
vary 4%→14% among sellers *in the same state* — proving seller execution is a
real, separable lever, not just geography.

## 9. Value-creation impact scenario

In [ ]:
nat = f['is_late'].mean()*100
fix = s[s['quadrant']=='2 Fix Immediately']
items_fix   = fix['items'].sum()
late_now    = (fix['late_pct']/100*fix['items']).sum()
late_target = nat/100*items_fix
avoided     = late_now - late_target
print(f'Fix-Immediately sellers: {len(fix)} | items {items_fix:,.0f} | revenue R$ {fix.revenue.sum():,.0f}')
print(f'Late deliveries avoided: {avoided:,.0f}')
print(f"Overall late %%: {f['is_late'].mean()*100:.2f}% -> {100*(f['is_late'].sum()-avoided)/len(f):.2f}%")
print(f'Relative reduction in late deliveries: {100*avoided/f["is_late"].sum():.1f}%')
print(f'Revenue exposure protected: R$ {avoided*f["revenue"].mean():,.0f}')

Fix-Immediately sellers: 609 | items 44,568 | revenue R$ 6,273,394
Late deliveries avoided: 1,999
Overall late %%: 7.91% -> 6.09%
Relative reduction in late deliveries: 22.9%
Revenue exposure protected: R$ 279,689


## 10. Key takeaways

1. **Reliability is good on average (92% on-time) but concentrated risk is the story.**
   609 high-volume "Fix Immediately" sellers carry ~41% of revenue at a ~16% late rate.
2. **The bottleneck is the last mile, not the seller's desk** — transit is ~74% of lead time.
3. **Geography compounds it** — cross-state shipping drives most lateness; consolidating
   fulfilment closer to demand (esp. the SP↔ rest-of-Brazil flows) is the structural fix.
4. **Late delivery is expensive** — it cuts average review from 4.3 to 2.6 and pushes 1–2★
   share from 9% to 53%, the leading indicator of churn.
5. **Sizing:** closing the Fix-Immediately gap to the national benchmark removes ~23% of all
   late deliveries and protects ~R$0.3M of directly exposed revenue — before second-order
   churn and reputation effects.
